In [42]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
spambase = fetch_ucirepo(id=94) 
  
# data (as pandas dataframes) 
X = spambase.data.features 
y = spambase.data.targets 
  
# metadata 
print(spambase.metadata) 
  
# variable information 
print(spambase.variables) 


{'uci_id': 94, 'name': 'Spambase', 'repository_url': 'https://archive.ics.uci.edu/dataset/94/spambase', 'data_url': 'https://archive.ics.uci.edu/static/public/94/data.csv', 'abstract': 'Classifying Email as Spam or Non-Spam', 'area': 'Computer Science', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 4601, 'num_features': 57, 'feature_types': ['Integer', 'Real'], 'demographics': [], 'target_col': ['Class'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1999, 'last_updated': 'Mon Aug 28 2023', 'dataset_doi': '10.24432/C53G6X', 'creators': ['Mark Hopkins', 'Erik Reeber', 'George Forman', 'Jaap Suermondt'], 'intro_paper': None, 'additional_info': {'summary': 'The "spam" concept is diverse: advertisements for products/web sites, make money fast schemes, chain letters, pornography...\n\nThe classification task for this dataset is to determine whether a given email is spam or not.\n\t\nOur collecti

In [43]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import pandas as pd



Use an existing package of your choice to train and test a logistic regression model on the _SPAMBASE_ dataset.

1. Split the original data into 75% for training and 25% for testing. Choose the training set at random. Train a logistic regression model on the training set and output the following **on the testing set**:
    1. **Confusion matrix**
    2. **Accuracy**, **Error**
    3. **Precision**, **Recall**, **F1 score**

In [44]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

model = LogisticRegression(max_iter=5000)
model.fit(X_train, y_train.values.ravel())

y_pred = model.predict(X_test)

In [45]:

cm = confusion_matrix(y_test, y_pred)

accuracy = accuracy_score(y_test, y_pred)
error = 1 - accuracy
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Confusion Matrix:\n", cm)
print("Accuracy:", accuracy)
print("Error:", error)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)


Confusion Matrix:
 [[651  25]
 [ 55 420]]
Accuracy: 0.9304952215464813
Error: 0.06950477845351866
Precision: 0.9438202247191011
Recall: 0.8842105263157894
F1 Score: 0.9130434782608695


Print the coefficients of the features in the model. Which features contribute mostly to the prediction? Which ones are positively
correlated and which ones are negatively correlated with the SPAM class?

In [46]:
coefficients = pd.Series(model.coef_[0], index=X.columns)

print("Top positive coefficients (most spam-indicating):")
print(coefficients.sort_values(ascending=False).head(10))

print("\nTop negative coefficients (most ham-indicating):")
print(coefficients.sort_values().head(10))

Top positive coefficients (most spam-indicating):
char_freq_$             3.495517
word_freq_000           2.136265
word_freq_remove        2.116497
word_freq_addresses     1.517192
char_freq_#             1.161521
word_freq_free          1.008968
word_freq_technology    0.984858
word_freq_credit        0.906423
word_freq_business      0.888698
word_freq_3d            0.755731
dtype: float64

Top negative coefficients (most ham-indicating):
word_freq_george       -4.190436
word_freq_hp           -1.819632
word_freq_conference   -1.798325
word_freq_project      -1.729896
word_freq_meeting      -1.536243
word_freq_cs           -1.352378
word_freq_edu          -1.343435
word_freq_data         -1.194612
word_freq_lab          -1.152946
char_freq_;            -1.010213
dtype: float64


Positively correlated with SPAM are words/symbols like char_freq_$, word_freq_remove, and word_freq_free, which increase the spam probability. Negatively correlated are words like word_freq_george, word_freq_meeting, and word_freq_edu, which indicate HAM emails.

Vary the decision threshold $T \in \{0.25,0.5,0.75,0.9\}$ and report for each value the model accuracy, precision, and recall. Comment on
how these metrics vary with the choice of threshold.

In [47]:
y_prob = model.predict_proba(X_test)[:, 1]

thresholds = [0.25, 0.5, 0.75, 0.9]

for T in thresholds:
    y_pred_T = (y_prob >= T).astype(int)

    acc = accuracy_score(y_test, y_pred_T)
    prec = precision_score(y_test, y_pred_T)
    rec = recall_score(y_test, y_pred_T)

    print("Threshold:", T)
    print("Accuracy:", acc)
    print("Precision:", prec)
    print("Recall:", rec)
    print()

Threshold: 0.25
Accuracy: 0.9070373588184187
Precision: 0.8382352941176471
Recall: 0.96

Threshold: 0.5
Accuracy: 0.9304952215464813
Precision: 0.9438202247191011
Recall: 0.8842105263157894

Threshold: 0.75
Accuracy: 0.8861859252823632
Precision: 0.9574468085106383
Recall: 0.7578947368421053

Threshold: 0.9
Accuracy: 0.8236316246741964
Precision: 0.9722222222222222
Recall: 0.5894736842105263



As the decision threshold increases, precision increases while recall decreases. Lower thresholds like 0.25 classify more emails as spam, catching almost all spam (high recall) but also including more false positives (lower precision). The higher thresholds like 0.9 are stricter, so fewer emails are predicted as spam leading to very high precision but many missed spam emails.